# 

In [1]:
# Imports
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
# Load Data

# Column names from the dataset's .names file
column_names = ['symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration','num-of-doors', 'body-style', 'drive-wheels', 'engine-location',
               'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-type', 'num-of-cylinders', 'engine-size', 'fuel-system',
               'bore', 'stroke', 'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'price']

relative_path = 'datasets/automobile/automobile/imports-85.data'

# Load data, recognising '?' as NaN
data = pd.read_csv(relative_path, header=None, names=column_names, sep=',', na_values='?')

print(f"Data loaded from {relative_path}.")
print(f"Initial shape: {data.shape}")

Data loaded from datasets/automobile/automobile/imports-85.data.
Initial shape: (205, 26)


In [3]:
# Data Pre-processing stuff

In [4]:
# Identifies Target and Handles Missing Targets

target_variable = 'price'
print(f"Target variable: {target_variable}")

# Drops rows where the target ('price') is missing
initial_rows = data.shape[0]
data.dropna(subset=[target_variable], axis=0, inplace=True)
print(f"Dropped {initial_rows - data.shape[0]} rows with missing target. New shape: {data.shape}")

Target variable: price
Dropped 4 rows with missing target. New shape: (201, 26)


In [5]:
# Handles Missing Features (this uses manual imputation based on Lab 2's SimpleImputer principles)

# Imputes numerical features with the mean
numerical_cols_to_impute = ['normalized-losses', 'bore', 'stroke', 'horsepower', 'peak-rpm']
for col in numerical_cols_to_impute:
    # Ensures column is numeric first
    data[col] = pd.to_numeric(data[col], errors='coerce')
    mean_val = data[col].mean()
    data[col].fillna(mean_val, inplace=True)
    print(f"Filled NaNs in numerical '{col}' with mean.")

# Imputes categorical features with the mode
categorical_cols_to_impute = ['num-of-doors']
for col in categorical_cols_to_impute:
    mode_val = data[col].mode()[0]
    data[col].fillna(mode_val, inplace=True)
    print(f"Filled NaNs in categorical '{col}' with mode.")

print("\nMissing values check after imputation:")
print(data.isnull().sum())

Filled NaNs in numerical 'normalized-losses' with mean.
Filled NaNs in numerical 'bore' with mean.
Filled NaNs in numerical 'stroke' with mean.
Filled NaNs in numerical 'horsepower' with mean.
Filled NaNs in numerical 'peak-rpm' with mean.
Filled NaNs in categorical 'num-of-doors' with mode.

Missing values check after imputation:
symboling            0
normalized-losses    0
make                 0
fuel-type            0
aspiration           0
num-of-doors         0
body-style           0
drive-wheels         0
engine-location      0
wheel-base           0
length               0
width                0
height               0
curb-weight          0
engine-type          0
num-of-cylinders     0
engine-size          0
fuel-system          0
bore                 0
stroke               0
compression-ratio    0
horsepower           0
peak-rpm             0
city-mpg             0
highway-mpg          0
price                0
dtype: int64


/var/folders/4h/rzfqsskd2bj_xd9b7xfgch_w0000gn/T/ipykernel_20581/3572761116.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(mean_val, inplace=True)
/var/folders/4h/rzfqsskd2bj_xd9b7xfgch_w0000gn/T/ipykernel_20581/3572761116.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

In [6]:
# Encoding Categorical Features (using pd.get_dummies, a pandas equivalent of OneHotEncoder we saw in Lab 2)
# Only did this because I'd have to use a ColumnTransformer to specify which columns to apply it to, while also leaving the numerical columns alone which is just more complex.

categorical_cols = data.select_dtypes(include='object').columns
print(f"\nCategorical columns to encode: {list(categorical_cols)}")
data_encoded = pd.get_dummies(data, columns=categorical_cols, drop_first=True)
print(f"Shape after encoding: {data_encoded.shape}")


Categorical columns to encode: ['make', 'fuel-type', 'aspiration', 'num-of-doors', 'body-style', 'drive-wheels', 'engine-location', 'engine-type', 'num-of-cylinders', 'fuel-system']
Shape after encoding: (201, 65)


In [7]:
# Final X and y Split
# Convert target 'price' to numeric (if it's not already)
data_encoded[target_variable] = pd.to_numeric(data_encoded[target_variable], errors='coerce')

X = data_encoded.drop(target_variable, axis=1)
y = data_encoded[target_variable]

print(f"\nFeatures (X) and Target (y) separated.")
print(f"X shape: {X.shape}, y shape: {y.shape}")


Features (X) and Target (y) separated.
X shape: (201, 64), y shape: (201,)


In [8]:
# 4. Model Definition & Evaluation Setup

In [9]:
# Defining Models (from Labs 1, 3, 4) 
# Going to use random_state for reproducible results
models_reg = {
    'Linear Regression': LinearRegression(),
    'SVR': SVR(), 
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42),
    'MLP Regressor': MLPRegressor(random_state=1, max_iter=1000)
}
print("\nModels defined.")


Models defined.


In [10]:
# Setup K-Fold Cross-Validation
K = 5 # I just chose this
kf = KFold(n_splits=K, shuffle=True, random_state=42)
print(f"Using K={K} for Cross-Validation.")

Using K=5 for Cross-Validation.


In [11]:
# Sets scoring metric from the Brief
scoring_metric = 'neg_mean_squared_error'
metric_name = 'Mean Squared Error (MSE)'
print(f"Task: regression, Evaluation Metric: {metric_name}")

Task: regression, Evaluation Metric: Mean Squared Error (MSE)


In [12]:
# Run K-Fold Cross-Validation

In [13]:
# We use a Pipeline to combine scaling and modeling,
# Going to use StandardScaler which is from Lab 2 and cross_val_score from Lab 5

results = {}

print(f"\n--- Running {K}-Fold Cross-Validation ---")

for model_name, model in models_reg.items():
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)           
    ])

    # Performs K-Fold Cross-Validation
    scores = cross_val_score(pipeline, X, y, cv=kf, scoring=scoring_metric, n_jobs=-1)

    mean_score = -np.mean(scores)
    std_dev_score = np.std(scores)

    results[model_name] = {'Mean MSE': mean_score, 'Std Dev MSE': std_dev_score}
    print(f"{model_name}: Mean MSE = {mean_score:,.2f} (+/- {std_dev_score:,.2f})")


--- Running 5-Fold Cross-Validation ---
Linear Regression: Mean MSE = 7,576,543.49 (+/- 2,080,645.04)
SVR: Mean MSE = 70,982,185.66 (+/- 46,347,253.67)
Decision Tree Regressor: Mean MSE = 7,179,196.21 (+/- 2,048,227.86)
MLP Regressor: Mean MSE = 178,590,650.39 (+/- 58,304,385.94)


/Users/richard/anaconda3/envs/MLE/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/richard/anaconda3/envs/MLE/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/richard/anaconda3/envs/MLE/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/richard/anaconda3/envs/MLE/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  war

In [14]:
# Collate Results for Report

In [15]:
results_df = pd.DataFrame(results).T
print("\n--- Summary Table for Report ---")
print(pd.DataFrame(results).T)
print("\nRegression task complete.")


--- Summary Table for Report ---
                             Mean MSE   Std Dev MSE
Linear Regression        7.576543e+06  2.080645e+06
SVR                      7.098219e+07  4.634725e+07
Decision Tree Regressor  7.179196e+06  2.048228e+06
MLP Regressor            1.785907e+08  5.830439e+07

Regression task complete.
